In [1]:
import pandas as pd

df = pd.read_csv("../data/PS_20174392719_1491204439457_log.csv")
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [2]:
print(f"Rows: {df.shape[0]:,}, Columns: {df.shape[1]}")
df.info()

Rows: 6,362,620, Columns: 11
<class 'pandas.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            str    
 2   amount          float64
 3   nameOrig        str    
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        str    
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), str(3)
memory usage: 534.0 MB


In [3]:
fraud_counts = df['isFraud'].value_counts()
fraud_pct = df['isFraud'].value_counts(normalize=True) * 100

print(fraud_counts)
print(f"\nFraud rate: {fraud_pct[1]:.4f}%")

isFraud
0    6354407
1       8213
Name: count, dtype: int64

Fraud rate: 0.1291%


In [4]:
df['type'].value_counts()

type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432
Name: count, dtype: int64

In [6]:
df[df['isFraud'] == 1]['type'].value_counts()

type
CASH_OUT    4116
TRANSFER    4097
Name: count, dtype: int64

## Key finding: fraud is confined to two transaction types

Fraud in this dataset occurs exclusively in **TRANSFER** and **CASH_OUT** transactions
(4,097 and 4,116 fraudulent cases respectively — 8,213 total). The other three types
(PAYMENT, CASH_IN, DEBIT) show **zero** fraud cases across 3.5M+ transactions combined.

Overall fraud rate: 0.1291% (8,213 / 6,362,620) — an extreme class imbalance that makes
plain accuracy a meaningless metric (predicting "not fraud" every time would still score
99.87%). Precision, recall, and AUC-PR are used instead throughout this project.

In [7]:
import sqlite3

conn = sqlite3.connect("../data/paysim.db")
df.to_sql("transactions", conn, if_exists="replace", index=False)

6362620

In [8]:
query = "SELECT type, COUNT(*) as cnt, SUM(isFraud) as fraud_cnt FROM transactions GROUP BY type"
pd.read_sql(query, conn)

,type,cnt,fraud_cnt
0,CASH_IN,1399284,0
1,CASH_OUT,2237500,4116
2,DEBIT,41432,0
3,PAYMENT,2151495,0
4,TRANSFER,532909,4097


In [9]:
feature_query = """
SELECT *,
    CASE WHEN ROUND(oldbalanceOrg - amount, 2) != ROUND(newbalanceOrig, 2)
         THEN 1 ELSE 0 END AS balance_mismatch_orig,
    CASE WHEN ROUND(oldbalanceDest + amount, 2) != ROUND(newbalanceDest, 2)
         THEN 1 ELSE 0 END AS balance_mismatch_dest,
    CASE WHEN oldbalanceOrg > 0 THEN amount / oldbalanceOrg ELSE 0 END AS amount_to_balance_ratio,
    CASE WHEN newbalanceOrig = 0 AND oldbalanceOrg > 0 THEN 1 ELSE 0 END AS emptied_account
FROM transactions
WHERE type IN ('TRANSFER', 'CASH_OUT')
"""
df_features = pd.read_sql(feature_query, conn)
print(df_features.shape)
df_features['isFraud'].value_counts()

(2770409, 15)


isFraud
0    2762196
1       8213
Name: count, dtype: int64

In [11]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Sort by time (step = hour of simulation) — critical, don't shuffle randomly
df_features = df_features.sort_values('step').reset_index(drop=True)

split_idx = int(len(df_features) * 0.8)
train = df_features.iloc[:split_idx]
test = df_features.iloc[split_idx:]

features = ['amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest',
            'newbalanceDest', 'balance_mismatch_orig', 'balance_mismatch_dest',
            'amount_to_balance_ratio', 'emptied_account']

X_train, y_train = train[features], train['isFraud']
X_test, y_test = test[features], test['isFraud']

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train fraud rate: {y_train.mean()*100:.4f}%")
print(f"Test fraud rate: {y_test.mean()*100:.4f}%")

Train: (2216327, 9), Test: (554082, 9)
Train fraud rate: 0.1784%
Test fraud rate: 0.7688%


## Note: temporal split reveals a shift in fraud rate over time

The 80/20 temporal split (sorted by `step`, not shuffled) shows a clear shift: the
training set has a fraud rate of **0.1784%**, while the test set — representing later
transactions — has a fraud rate of **0.7688%**, over 4x higher.

This is exactly why a temporal split was used instead of a random one. A random split
would have blended early and late transactions together, hiding this shift and giving
an artificially easier (and less realistic) evaluation. Here, the model is being tested
on a period with meaningfully more fraud than it saw during training — a harder,
more honest test of whether it generalizes rather than just memorizes patterns from
a single time window.